# Using the Claude and Codex models

The live check for spec 0012: one call per model, through its controller, on a
one-line passage with the purpose/risk prompt. Each call runs on your
subscription, never an API key, and spends a little of its quota. Claude goes
through Claude Code's login (`claude -p`), about 1.6k input tokens. Codex calls
OpenAI's Responses API with a Sign in with ChatGPT token that bills your
ChatGPT plan: the prompt alone, no tools, so about the prompt's own size.

Settings come from `configs/claude.toml` and `configs/codex.toml`; pass an
argument to `ClaudeModel(...)` / `CodexModel(...)` to override one.

In [1]:
from engine.controllers.claude_controller import ClaudeController
from engine.controllers.codex_controller import CodexController
from engine.models.base_model import Prompt
from engine.models.claude_model import ClaudeModel
from engine.models.codex_model import CodexModel
from engine.repositories.prompts import loader as prompts

spec = prompts.load("purpose_risk")
passage = "supports healthy joints. do not take if pregnant. keep out of reach of children."
prompt = Prompt(system=spec.system, user=passage, stop=spec.stop, max_tokens=spec.max_tokens)

## Sign in to ChatGPT (Codex only, once per machine)

Run the next cell only when Codex's `ready()` asks you to sign in: the first
time on this machine, or when the sign-in has expired or was revoked. It
prints a link; open it in your browser and approve "supplement-kg judge". The
redirect goes to `127.0.0.1:1455`, so over SSH, port 1455 must be forwarded
(VSCode's Ports panel). The credentials are stored in
`~/.config/supplement-kg/chatgpt-plan/`, readable only by you; the refresh
token then renews itself for 30 days at a time.

In [2]:
SIGN_IN = False  # True when ready() below asks you to sign in
if SIGN_IN:
    CodexModel().sign_in()

## Ready

Claude: no model call; the binary runs and the CLI is logged in. A failure
names the fix -- `claude login`, or the toml line whose binary moved. Codex:
the credentials are loaded and one test call (~20 tokens) is made, because only
a completed call shows the sign-in still works. If it fails with "sign in
again", run the sign-in cell above with `SIGN_IN = True`.

In [3]:
claude = ClaudeController()
codex = CodexController()
claude.model.ready()
codex.model.ready()
{"claude": (claude.model.binary, claude.model.model, claude.model.effort),
 "codex": (codex.model.credentials_dir, codex.model.model, codex.model.effort)}

{'claude': ('/home/ge74wan/.local/bin/claude', 'claude-sonnet-5-5', 'medium'),
 'codex': ('/home/ge74wan/.config/supplement-kg/chatgpt-plan',
  'gpt-6.1-terra',
  'medium')}

## One call each

Expected from both: `P: supports healthy joints` / `R: do not take if
pregnant` / `END`, `ok=True` on the first attempt. `Generation.model` says
which model answered: `<id>@claude-code-<version>` for Claude,
`<id>@responses` for Codex.

In [4]:
for name, controller in [("claude", claude), ("codex", codex)]:
    [outcome] = controller.handle([prompt], checks=spec.checks)
    g = outcome.generation
    print(f"== {name}: ok={outcome.ok} attempts={outcome.attempts} failed={outcome.failed}")
    print(f"   model={g.model} latency={g.latency_ms} ms usage={g.usage} stop={g.stop_reason!r}")
    print(g.text)

== claude: ok=True attempts=1 failed=[]
   model=claude-sonnet-5-5@claude-code-2.1.311 latency=5148 ms usage=Usage(prompt_tokens=1263, completion_tokens=24) stop='end_turn'
P: supports healthy joints
R: do not take if pregnant
END
== codex: ok=True attempts=1 failed=[]
   model=gpt-6.1-terra@responses latency=4249 ms usage=Usage(prompt_tokens=546, completion_tokens=19) stop='completed'
P: supports healthy joints
R: do not take if pregnant
END


## Close

Cancels anything queued and stops running calls (Claude's processes are
killed, Codex's streams cut). Safe to run twice.

In [5]:
claude.close()
codex.close()